In [30]:

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

print("=== BÀI 3: KHAI THÁC LUẬT KẾT HỢP (D2 & D3) ===")

try:
    # --- 1. XỬ LÝ D2 (HOTEL BOOKING) ---
    print("\n[1] Đang xử lý bộ dữ liệu D2 (Hotel Bookings)...")
    df2 = pd.read_csv('hotel_bookings.csv', nrows=5000)

    df2['lead_time_bin'] = pd.qcut(df2['lead_time'], q=3, labels=['Short', 'Medium', 'Long'])
    d2_basket = pd.get_dummies(df2[['hotel', 'meal', 'customer_type', 'lead_time_bin']]).astype(bool)

    freq2_c1 = apriori(d2_basket, min_support=0.1, use_colnames=True)
    rules2_c1 = association_rules(freq2_c1, metric="confidence", min_threshold=0.5)

    rules2_final = rules2_c1[rules2_c1['lift'] > 1.0].sort_values('lift', ascending=False).reset_index(drop=True)

    print(f"-> D2: Tìm thấy {len(rules2_final)} luật có giá trị (Lift > 1.0).")
    print("TOP 3 LUẬT MẠNH NHẤT CỦA D2:")
    print(rules2_final[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(3))


    # --- 2. XỬ LÝ D3 (GROCERIES) ---
    print("\n" + "="*50)
    print("\n[2] Đang xử lý bộ dữ liệu D3 (Groceries)...")
    with open('Groceries.csv', 'r') as f:
        transactions = [line.strip().split(',') for line in f.readlines()]

    te = TransactionEncoder()
    d3_basket = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)

    # NỚI LỎNG NGƯỠNG: Hạ Support xuống 0.005 (0.5%) và Confidence xuống 0.1 (10%)
    freq3_c1 = apriori(d3_basket, min_support=0.005, use_colnames=True)
    rules3_c1 = association_rules(freq3_c1, metric="confidence", min_threshold=0.1)

    # NỚI LỎNG BỘ LỌC LIFT: Chỉ cần Lift > 1.2 là lấy
    rules3_final = rules3_c1[rules3_c1['lift'] > 1.2].sort_values('lift', ascending=False).reset_index(drop=True)

    print(f"-> D3: Tìm thấy {len(rules3_final)} luật có giá trị (Lift > 1.2).")
    print("TOP 3 LUẬT MẠNH NHẤT CỦA D3:")
    print(rules3_final[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(3))

except FileNotFoundError as e:
    print(f"LỖI: Không tìm thấy file dữ liệu. Hãy kiểm tra lại thư mục data/raw/. Chi tiết: {e}")
except Exception as e:
    print(f"LỖI HỆ THỐNG: {e}")

=== BÀI 3: KHAI THÁC LUẬT KẾT HỢP (D2 & D3) ===

[1] Đang xử lý bộ dữ liệu D2 (Hotel Bookings)...
-> D2: Tìm thấy 33 luật có giá trị (Lift > 1.0).
TOP 3 LUẬT MẠNH NHẤT CỦA D2:
                                 antecedents  \
0  (hotel_Resort Hotel, lead_time_bin_Short)   
1                      (lead_time_bin_Short)   
2                      (lead_time_bin_Short)   

                                         consequents  support  confidence  \
0                 (meal_BB, customer_type_Transient)   0.2372    0.698469   
1                 (meal_BB, customer_type_Transient)   0.2372    0.698469   
2  (meal_BB, hotel_Resort Hotel, customer_type_Tr...   0.2372    0.698469   

       lift  
0  1.259864  
1  1.259864  
2  1.259864  


[2] Đang xử lý bộ dữ liệu D3 (Groceries)...
-> D3: Tìm thấy 0 luật có giá trị (Lift > 1.2).
TOP 3 LUẬT MẠNH NHẤT CỦA D3:
Empty DataFrame
Columns: [antecedents, consequents, support, confidence, lift]
Index: []
